<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">El Fine Tuning Parcial y Total</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">No siempre conviene adaptar todo el modelo preentrenado del mismo modo. Este notebook compara el ajuste parcial y el ajuste total como dos estrategias de adaptación con costes y riesgos distintos.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Decidir cuánto del modelo se reentrena afecta a la estabilidad, al coste computacional y a la capacidad de adaptación al nuevo dominio.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A diferenciar fine tuning parcial y total, y a entender cuándo puede tener sentido cada enfoque.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Reentrenar más capas no garantiza un resultado mejor; depende de los datos disponibles y de la cercanía entre tareas.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Fine tuning significa seguir entrenando un modelo preexistente ajustando algunas o todas sus capas.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Carga del modelo preentrenado
"""Este bloque carga un modelo preentrenado utilizando torchvision.
Este bloque importa "torchvision.models" para acceder a modelos preentrenados disponibles en la librería torchvision.
Aplicamos la función "models.resnet18()" a la variable "modelo" para obtener un modelo preentrenado en ImageNet.
El parámetro "pretrained=" indica que se cargan los pesos aprendidos durante el preentrenamiento; esta forma está obsoleta en versiones recientes.
La variable "modelo" almacena la red que servirá como base para aplicar finetuning parcial o total."""

import torchvision.models as models

modelo = models.resnet18(pretrained=True)

modelo # Muestra la arquitectura del modelo preentrenado cargado.


In [ ]:
# Bloque 2 — Fine‑tuning parcial: congelación de capas
"""Este bloque congela todos los parámetros del modelo antes de sustituir la capa final.
Utilizamos un bucle "for" para recorrer cada parámetro del modelo mediante el método ".parameters()".
Dentro del bucle asignamos el valor "False" a la propiedad ".requires_grad" de cada parámetro con la expresión "p.requires_grad = False".

La variable "p" hace referencia a cada uno de los parámetros del modelo durante la iteración.
La propiedad ".requires_grad" indica si se calculan gradientes para el parámetro.
El modelo permanece en modo entrenamiento: las estadísticas móviles de BatchNorm aún pueden cambiar durante el forward pass."""
                      
for p in modelo.parameters():
    p.requires_grad = False

# Todos los parámetros quedan congelados; en el siguiente bloque se crea una nueva capa final entrenable.


In [ ]:
# Bloque 3 — Sustitución de la capa final para la tarea objetivo
"""Este bloque reemplaza la capa final del modelo para adaptarlo a una nueva tarea.
Este bloque importa "torch.nn" para trabajar con capas y módulos de redes neuronales. 

Aplicamos la clase "nn.Linear()" a la variable "modelo.fc" para reemplazar la capa fully connected original con una nueva capa lineal 
específica de la tarea objetivo.

La propiedad ".fc" de la variable "modelo" hace referencia a la capa fully connected que estamos reemplazando.
El parámetro "in_features=" indica el número de características de entrada de la capa fully connected original, que en el caso de resnet18 es 512.
El parámetro "out_features=" indica el número de características de salida de la nueva capa fully connected, que debe coincidir con el número
de clases de la tarea objetivo.

El atributo "modelo.fc" contiene la nueva capa lineal, inicializada sin entrenar y con parámetros entrenables.
La arquitectura resnet18 utiliza una capa fully connected llamada fc como salida."""

import torch.nn as nn

modelo.fc = nn.Linear(in_features=512, out_features=4)

modelo.fc # Muestra la nueva capa fully connected del modelo.


In [ ]:
# Bloque 4 — Fine‑tuning total: habilitación del entrenamiento completo
"""Este bloque habilita los gradientes para un posible fine-tuning total del modelo.
Utilizamos un bucle "for" para recorrer cada parámetro del modelo mediante el método ".parameters()".
Dentro del bucle asignamos el valor "True" a la propiedad ".requires_grad" de cada parámetro con la expresión "p.requires_grad = True".

La variable "p" hace referencia a cada uno de los parámetros del modelo durante la iteración.
La propiedad ".requires_grad" indica si se calculan gradientes para el parámetro.

Además, verificamos si todos los parámetros del modelo están habilitados para el entrenamiento completo mediante la expresión "all(p.requires_grad for p in modelo.parameters())".
La variable "parametros_entrenables" almacena un valor booleano que indica si todos los parámetros del modelo están habilitados para el entrenamiento completo.
La operación permite calcular gradientes para todos los parámetros, pero no entrena el modelo por sí sola."""

for p in modelo.parameters():
    p.requires_grad = True

parametros_entrenables = all(p.requires_grad for p in modelo.parameters())

parametros_entrenables # Muestra si todos los parámetros están habilitados para el entrenamiento completo.


In [ ]:
# Bloque 5 — Creación de una entrada sintética
"""Este bloque crea un tensor que simula una imagen de entrada.
Este bloque importa "torch" para trabajar con tensores en PyTorch.

Aplicamos la función "torch.randn()" a la variable "entrada" para crear un tensor que simula una imagen de entrada con valores aleatorios.
El primer argumento "1" indica el tamaño del lote de entrada (en este caso, 1 imagen por lote).
El segundo argumento "3" indica el número de canales de la imagen (en este caso, 3 canales RGB).
El tercer argumento "224" indica la altura de la imagen en píxeles.
El cuarto argumento "224" indica la anchura de la imagen en píxeles.

La variable "entrada" contiene un lote de tamaño 1 con una imagen de 3 canales y resolución 224x224.
La forma es compatible con ResNet18, pero los valores aleatorios no representan una imagen real preprocesada para los pesos preentrenados."""

import torch

entrada = torch.randn(1, 3, 224, 224)

entrada # Muestra el tensor que simula la imagen de entrada.


In [ ]:
# Bloque 6 — Forward pass del modelo en fine‑tuning total
"""Este bloque ejecuta el forward pass del modelo con fine-tuning total habilitado.
Asignamos la llamada a la instancia de la clase a la variable "salida" mediante la expresión "modelo(entrada)".
El argumento "entrada" es el tensor que contiene la imagen de entrada que se va a procesar por el modelo.
La variable "salida" contiene los logits generados por la nueva capa final.
El resultado muestra una salida con cuatro valores; el ejemplo habilita los gradientes, pero no realiza entrenamiento."""

salida = modelo(entrada)

salida # Muestra los logits generados tras habilitar el entrenamiento completo.


In [ ]:
# Bloque 7 — Resultados finales
"""Este bloque organiza los resultados del proceso de fine-tuning parcial y total.
La variable "resultado" almacena en forma de diccionario el estado de los parámetros entrenables, la estructura de la capa final y la salida generada.
La estructura muestra el estado tras habilitar todos los parámetros, pero no compara resultados de entrenamiento parcial y total."""

resultado = {
    "entrenamiento_total_habilitado": parametros_entrenables,
    "estructura_capa_final": str(modelo.fc),
    "salida": salida.tolist()
}

resultado # Muestra el diccionario con los resultados de la configuración de fine-tuning.

<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto que adaptar un modelo preentrenado implica decidir cuánto conocimiento previo se conserva y cuánto se reajusta.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">El grado de fine tuning debe elegirse según datos, dominio y riesgo de sobreajuste.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Desbloquear capas sin un criterio claro y perder la ventaja que ofrecía el preentrenamiento.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en la Aplicación en la Clasificación de Imágenes para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en la Aplicación en la Clasificación de Imágenes y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">11_Transfer_learning/04_La_aplicacion_en_la_clasificacion_de_imagenes.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">